Install libraries

In [1]:
# INSTALL
!pip install -q --no-warn-script-location langchain-groq langgraph tavily-python sympy matplotlib numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.9 MB/s eta 0:00:00


Imports libraries

In [2]:
# CELL 1: IMPORTS AND INITIAL SETUP
import os
import sympy as sp
from sympy import sympify, solve, diff, integrate, limit, Eq, latex, simplify
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import base64
import warnings
warnings.filterwarnings('ignore')

# LangChain / LangGraph imports
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated, Sequence

print("All imports successful")

All imports successful


SECTION 1: API Key Loading

In [3]:
# CELL 2 API KEYS
print("API Key Setup")

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
except:
    GROQ_API_KEY = None
    TAVILY_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = input("Enter your Groq API Key: ").strip()
if not TAVILY_API_KEY:
    TAVILY_API_KEY = input("Enter your Tavily API Key: ").strip()

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

API Key Setup


SECTION 2: AdvancedMathSolver (step by step, level-aware)

In [4]:
# CELL 3: ADVANCED MATH SOLVER (Concise Medium-Level Solutions)
class AdvancedMathSolver:
    """Solves equations with step-by-step explanations, adjustable complexity."""

    def __init__(self):
        self.symbols = {'x': sp.Symbol('x')}

    def _parse_equation(self, eq_str: str):
        if '=' in eq_str:
            left, right = eq_str.split('=')
            return Eq(sympify(left), sympify(right))
        else:
            return Eq(sympify(eq_str), 0)

    def solve_with_steps(self, equation: str, variable: str = 'x', level: str = 'medium'):
        """
        Returns dict with final_answer, steps (list), latex_answer.
        Level: 'easy' (minimal), 'medium' (balanced), 'hard' (detailed)
        """
        try:
            var = sp.Symbol(variable)
            eq = self._parse_equation(equation)
            solutions = solve(eq, var, dict=True)
            if not solutions:
                return {"error": "No solution found", "steps": [], "final_answer": None}

            steps = []
            if level == 'easy':
                steps.append(f"Solve {latex(eq)} for {variable}")
            else:
                steps.append(f"Equation: {latex(eq)}")
                if level == 'hard':
                    steps.append(f"Simplify: {latex(simplify(eq.lhs - eq.rhs))} = 0")

            # Format solutions
            sol_strs = [f"{variable} = {latex(sol[var])}" for sol in solutions]
            final_answer = ", ".join(sol_strs) if len(sol_strs) > 1 else sol_strs[0]
            if level == 'easy':
                steps.append(f"Solution: {final_answer}")
            elif level == 'medium':
                steps.append(f"Solution: {final_answer}")
                if len(solutions) > 1:
                    steps.append(f"Multiple solutions: {final_answer}")
            else:
                steps.append(f"Final answer: {final_answer}")
                if len(solutions) > 1:
                    steps.append(f"All solutions: {final_answer}")

            return {
                "final_answer": final_answer,
                "steps": steps,
                "latex_answer": f"\\boxed{{{final_answer}}}",
                "solutions": solutions
            }
        except Exception as e:
            return {"error": str(e), "steps": [], "final_answer": None}

SECTION 3: LangChain Tools (math_solver + web_search)

In [5]:
# CELL 4: TOOLS FOR THE AGENT
from tavily import TavilyClient

# Global solver instance
_math_solver_global = AdvancedMathSolver()

@tool
def math_solver_tool(equation: str, level: str = "medium") -> str:
    """
    Solves a mathematical equation with step-by-step explanation.
    Input: equation as string (e.g., "2^(x+1) + 2^(x-1) = 40")
    Level can be: easy, medium, hard, very hard.
    """
    res = _math_solver_global.solve_with_steps(equation, level=level)
    if "error" in res:
        return f"Math error: {res['error']}"
    steps_text = "\n\n".join(res["steps"])
    return f"**Final Answer:** {res['final_answer']}\n\n**Step-by-Step Solution:**\n{steps_text}"

@tool
def web_search_tool(query: str) -> str:
    """Search the web for additional math concepts or formulas."""
    try:
        client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY", ""))
        if not client.api_key:
            return "Tavily API key missing – cannot search."
        response = client.search(query, max_results=3, search_depth="basic")
        snippets = [f"• {r['content'][:200]}..." for r in response.get('results', [])]
        return "\n".join(snippets) if snippets else "No results."
    except Exception as e:
        return f"Search error: {str(e)}"

SECTION 4: LLM & Agent Setup

In [6]:
# CELL 5: INITIALISE LLM AND BUILD LANGGRAPH AGENT
print("Connecting to Groq LLM...")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=2000,
    api_key=GROQ_API_KEY
)

tools = [math_solver_tool, web_search_tool]
llm_with_tools = llm.bind_tools(tools)

# Define agent state
class AgentStateDict(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def ai_node(state: AgentStateDict):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_executor = ToolNode(tools)

# Build graph
workflow = StateGraph(AgentStateDict)
workflow.add_node("ai", ai_node)
workflow.add_node("tools", tool_executor)

workflow.add_edge(START, "ai")
workflow.add_conditional_edges("ai", tools_condition, {"tools": "tools", END: END})
workflow.add_edge("tools", "ai")

memory_saver = MemorySaver()
math_agent = workflow.compile(checkpointer=memory_saver)

print("Agent ready!")

Connecting to Groq LLM...
Agent ready!


SECTION 5: System Prompt & Chat Helper

In [7]:
# CELL 6: SYSTEM PROMPT AND CHAT FUNCTION
system_prompt = """
You are **MathWizard**, a friendly and highly skilled mathematics tutor.
- Always give **step-by-step** solutions (like Gemini/AI Studio).
- After the steps, put the **final answer inside a box**: \\boxed{answer}.
- If the user asks for a specific difficulty (easy/medium/hard/very hard), adjust your explanation depth accordingly.
- Mix English and simple Hindi when helpful.
- Keep explanations clear, concise, and engaging.
"""

def run_agent(user_input: str, history_list):
    """Process user input and return updated history."""
    if not user_input or not user_input.strip():
        return history_list, history_list

    # Prepare messages (system + conversation)
    messages = [SystemMessage(content=system_prompt)]
    for human, ai in history_list:
        messages.append(HumanMessage(content=human))
        messages.append(AIMessage(content=ai))
    messages.append(HumanMessage(content=user_input.strip()))

    config = {"configurable": {"thread_id": "mathwizard_session"}}
    final_response = ""
    try:
        for event in math_agent.stream({"messages": messages}, config, stream_mode="values"):
            last_msg = event["messages"][-1]
            if isinstance(last_msg, AIMessage) and last_msg.content:
                final_response = last_msg.content
    except Exception as e:
        final_response = f" Error: {str(e)}"

    history_list.append((user_input, final_response))
    return history_list, history_list

def reset_chat():
    return [], []

SECTION 6: Gradio Interface

In [8]:
# CELL 7: GRADIO UI
import gradio as gr

with gr.Blocks(title="MathWizard – Advanced Math Solver", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # **MathWizard** - Solve Any Equation with Step-by-Step Explanations
    *Like Google Gemini / AI Studio - but inside your own AI agent.*
    """)

    chatbot = gr.Chatbot(
        height=600,
        label="Conversation",
        bubble_full_width=False,
        show_copy_button=True,
        avatar_images=(None, None)
    )

    with gr.Row():
        user_msg = gr.Textbox(
            placeholder="Example: Solve 2^(x+1) + 2^(x-1) = 40 (medium level)",
            label="Your Math Question",
            lines=2,
            autofocus=True
        )

    with gr.Row():
        send_btn = gr.Button("Solve", variant="primary")
        clear_btn = gr.Button("Clear Chat", variant="secondary")

    gr.Markdown("""
    **Tips:**
    - Add difficulty: *"Solve ... (hard)"*
    - Ask for derivatives, integrals, or any algebra.
    - The agent will show step-by-step reasoning and a final boxed answer.
    """)

    # Event handlers
    send_btn.click(
        fn=run_agent,
        inputs=[user_msg, chatbot],
        outputs=[chatbot, chatbot]
    ).then(
        fn=lambda: "", inputs=None, outputs=user_msg
    )

    user_msg.submit(
        fn=run_agent,
        inputs=[user_msg, chatbot],
        outputs=[chatbot, chatbot]
    ).then(
        fn=lambda: "", inputs=None, outputs=user_msg
    )

    clear_btn.click(
        fn=reset_chat,
        inputs=None,
        outputs=[chatbot]
    )

# Launch
if __name__ == "__main__":
    demo.launch(share=True, debug=False)